# Full Pipeline: Load → Clean → Merge

Run cells top to bottom, in order, in a **fresh kernel** (Kernel → Restart, if you've run anything before this).

Requires `data/raw/KiDatabase.csv` and `data/raw/meddra_freq.tsv.gz` to already be downloaded.

In [1]:
import sys
sys.path.insert(0, '../src')

# Force-clear any stale cached imports from earlier sessions
for mod in ['config', 'load_data', 'join_data']:
    if mod in sys.modules:
        del sys.modules[mod]

import config
print('RAW_DIR:', config.RAW_DIR)
print('Type:', type(config.RAW_DIR))
print('Exists:', config.RAW_DIR.exists())

RAW_DIR: C:\project-repo-dsv\repo\data\raw
Type: <class 'pathlib.WindowsPath'>
Exists: True


**Check the output above before continuing.** `RAW_DIR` must be a `Path` object (not a plain string), and `Exists` must say `True`. If not, stop here and fix `src/config.py` before running anything else.

In [2]:
from load_data import load_pdsp, clean_pdsp, load_sider_freq
import pandas as pd

pdsp_raw = load_pdsp()
print(f'Loaded {len(pdsp_raw)} raw PDSP rows')
pdsp_raw.head()

Loaded 91868 raw PDSP rows


,Number,Name,Unigene,Ligand ID,Ligand Name,SMILES,CAS,NSC,Hotligand,species,source,ki Note,ki Val,Reference,Link
0,1,5-HT2C,HTR2C,172,5-Methoxytryptamine,COC1=CC2=C(C=C1)NC=C2CCN,608-07-1,56422.0,3H-MESULERGINE,PIG,Choroid Plexus,NaN,23.4,"HOYER, 1989 (J.R.R.)",NaN
1,2,5-HT2C,HTR2C,172,5-Methoxytryptamine,COC1=CC2=C(C=C1)NC=C2CCN,608-07-1,56422.0,3H-MESULERGINE,PIG,Choroid Plexus,NaN,23.4,"HOYER, 1988",NaN
2,3,5-HT2C,HTR2C,172,5-Methoxytryptamine,COC1=CC2=C(C=C1)NC=C2CCN,608-07-1,56422.0,3H-MESULERGINE,PIG,Choroid Plexus,NaN,25.1,"HOYER, 1988 or 9 (book)",NaN
3,4,5-HT2C,HTR2C,172,5-Methoxytryptamine,COC1=CC2=C(C=C1)NC=C2CCN,608-07-1,56422.0,3H-MESULERGINE,PIG,Choroid Plexus,NaN,43.7,"MARKSTEIN ET AL, 1986",NaN
4,5,5-HT2C,HTR2C,172,5-Methoxytryptamine,COC1=CC2=C(C=C1)NC=C2CCN,608-07-1,56422.0,3H-MESULERGINE,PIG,Choroid Plexus,NaN,43.8,"ENGEL ET AL, 1986",NaN


In [3]:
pdsp_clean = clean_pdsp(pdsp_raw)
print(f'{len(pdsp_clean)} cleaned substance-receptor pairs')
pdsp_clean.to_csv(config.PROCESSED_DIR / 'pdsp_clean.csv', index=False)
pdsp_clean.head()

68956 cleaned substance-receptor pairs


,Ligand Name,Receptor,Ki_value,pKi
0,"1-( 2,5-Dimet hoxy-4-propionylphenyl)-2-amino...",5-HT2,7550.0,5.122053
1,"1-(5-Chloropyridin-2-yl)-2-(2,4-dichloropheny...",CANNABINOID CB2,758.0,6.120331
2,"1-(Aminomethyl)dibenzo[b,e]bicyclo[2.2.2]octa...",5-HT2A,770.0,6.113509
3,"1-(Aminomethyl)dibenzo[b,e]bicyclo[2.2.2]octa...",5-HT6,169.0,6.772113
4,"2-(4-ethyl-2,5- dimethoxyphenyl)-N-[(2-methox...",Norepinephrine transporter,3000.0,5.522879


In [4]:
sider = load_sider_freq()
print(f'Loaded {len(sider)} SIDER frequency rows')
sider.to_csv(config.PROCESSED_DIR / 'sider_freq.csv', index=False)
sider.head()

Loaded 291632 SIDER frequency rows


,stitch_id_flat,stitch_id_stereo,umls_cui_from_label,placebo,frequency,lower,upper,meddra_type,umls_cui_from_meddra,side_effect_name
0,CID100000085,CID000010917,C0000737,NaN,21%,0.21,0.21,LLT,C0000737,Abdominal pain
1,CID100000085,CID000010917,C0000737,NaN,21%,0.21,0.21,PT,C0000737,Abdominal pain
2,CID100000085,CID000010917,C0000737,NaN,21%,0.21,0.21,PT,C0687713,Gastrointestinal pain
3,CID100000085,CID000010917,C0000737,NaN,5%,0.05,0.05,LLT,C0000737,Abdominal pain
4,CID100000085,CID000010917,C0000737,NaN,5%,0.05,0.05,PT,C0000737,Abdominal pain


## Merge PDSP + SIDER on PubChem CID

This filters PDSP down to a curated ~40-drug list first, so the PubChem lookup takes seconds instead of hours. Requires internet access.

In [5]:
from join_data import filter_to_relevant_drugs, merge_pdsp_sider

pdsp_filtered = filter_to_relevant_drugs(pdsp_clean)
pdsp_filtered.head()

Filtered 68956 rows -> 2341 rows (102 unique ligands)


,Ligand Name,Receptor,Ki_value,pKi
22,"((Â±)-3,4-Methylenedioxymethamphetamine",Nicotinic Alpha2Beta2,10000.0,5.000000
186,(+)-Amphetamine,Norepinephrine transporter,50.0,7.301030
604,(-)-Amphetamine,Norepinephrine transporter,90.0,7.045757
11244,"(Â±)-2,4,5-Trimethoxamphetamine",5-HT1A,10000.0,5.000000
11245,"(Â±)-2,4,5-Trimethoxamphetamine",5-HT1B,10000.0,5.000000


In [6]:
merged = merge_pdsp_sider(pdsp_filtered, sider)
merged.to_csv(config.PROCESSED_DIR / 'merged_receptor_side_effects.csv', index=False)

print(f'Merged dataset: {len(merged)} rows')
print(f'Unique drugs matched: {merged["pubchem_cid"].nunique()}')
merged.head()

Looking up PubChem CIDs for 102 unique ligand names...
  ...25/102 done
  ...50/102 done
  ...75/102 done
  ...100/102 done
Merged dataset: 1266929 rows
Unique drugs matched: 21


,Ligand Name,Receptor,Ki_value,pKi,pubchem_cid,stitch_id_flat,stitch_id_stereo,umls_cui_from_label,placebo,frequency,lower,upper,meddra_type,umls_cui_from_meddra,side_effect_name
0,AMITRIPTYLINE,5-HT Transporter,26.8,7.571865,2160.0,CID100002160,CID000002160,C0003864,NaN,postmarketing,0.0,0.001,LLT,C0003864,Arthritis
1,AMITRIPTYLINE,5-HT Transporter,26.8,7.571865,2160.0,CID100002160,CID000002160,C0003864,NaN,postmarketing,0.0,0.001,PT,C0003864,Arthritis
2,AMITRIPTYLINE,5-HT Transporter,26.8,7.571865,2160.0,CID100002160,CID000002160,C0015967,NaN,postmarketing,0.0,0.001,LLT,C0015967,Body temperature increased
3,AMITRIPTYLINE,5-HT Transporter,26.8,7.571865,2160.0,CID100002160,CID000002160,C0015967,NaN,postmarketing,0.0,0.001,PT,C0015967,Body temperature increased
4,AMITRIPTYLINE,5-HT Transporter,26.8,7.571865,2160.0,CID100002160,CID000002160,C0026837,NaN,postmarketing,0.0,0.001,LLT,C0026837,Muscle rigidity


## Sanity check: which relevant drugs actually matched?

In [7]:
matched_names = sorted(merged['Ligand Name'].unique())
print(f'{len(matched_names)} drugs matched in both PDSP and SIDER:')
for name in matched_names:
    print(' -', name)

23 drugs matched in both PDSP and SIDER:
 - AMITRIPTYLINE
 - ARIPIPRAZOLE
 - Alprazolam
 - Amphetamine
 - BUPROPION
 - BUSPIRONE
 - CARBAMAZEPINE
 - CITALOPRAM
 - CLOMIPRAMINE
 - DIAZEPAM
 - DOXEPIN
 - Desvenlafaxine
 - Doxepine
 - FLUOXETINE
 - HALOPERIDOL
 - METHYLPHENIDATE
 - O-Desmethylvenlafaxine
 - Quetiapine
 - RISPERIDONE
 - TRAZODONE
 - Venlafaxine
 - Zolpidem
 - mirtazapine
